In [1]:
import sys
sys.path.append("../..")
import importlib
import clean_data

importlib.reload(clean_data)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path 
from catboost import CatBoostClassifier, Pool 

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, log_loss, f1_score, balanced_accuracy_score, accuracy_score

from clean_data import feature_engineering_days

In [2]:
df = pd.read_csv("../../data/austin/raw/aac_intakes.csv")
outcomes = pd.read_csv("../../data/austin/raw/aac_intakes_outcomes.csv")

df["datetime"] = pd.to_datetime(df["datetime"])
outcomes["intake_datetime"] = pd.to_datetime(outcomes["intake_datetime"])

df = df.merge(
    outcomes[["animal_id_intake", "intake_datetime", "time_in_shelter_days"]],
    left_on=["animal_id", "datetime"],
    right_on=["animal_id_intake", "intake_datetime"],
    how="inner"
)

df = df[df["animal_type"] == "Cat"].copy()

In [3]:
df = feature_engineering_days(df)

df_cats = df[df["animal_type"] == "Cat"].copy()
df_cats = df_cats.sort_values("intake_datetime").reset_index(drop=True)

bins = [-np.inf, 4, 10, np.inf]
df_cats["target"] = pd.cut(
    df_cats["time_in_shelter_days"], bins=bins, labels=[0, 1, 2]
).astype(int)

In [4]:
drop_cols = [
    "target", "time_in_shelter_days", "time_in_shelter",
    "outcome_type", "outcome_subtype", "sex_upon_outcome",
    "intake_datetime", "animal_type", "young_and_cat", "datetime", "animal_id_intake"
]

X = df_cats.drop(columns=drop_cols, errors="ignore")
y = df_cats["target"]

#categorical features which will be available
cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
X[cat_features] = X[cat_features].fillna("Unknown").astype(str)

#70/15/15
n = len(X)
i1, i2 = int(n * 0.70), int(n * 0.85)

X_train, X_val, X_test = X.iloc[:i1], X.iloc[i1:i2], X.iloc[i2:]
y_train, y_val, y_test = y.iloc[:i1], y.iloc[i1:i2], y.iloc[i2:]

print("Split:", len(X_train), len(X_val), len(X_test))

Split: 20526 4399 4399


In [5]:
model = CatBoostClassifier(
    iterations=1200, learning_rate=0.03, depth=6,
    loss_function="MultiClass", random_seed=42, verbose=100
)

model.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100,
    use_best_model=True
)

0:	learn: 1.0799878	test: 1.0821649	best: 1.0821649 (0)	total: 313ms	remaining: 6m 14s
100:	learn: 0.7275536	test: 0.8130394	best: 0.8130394 (100)	total: 10.1s	remaining: 1m 49s
200:	learn: 0.7012655	test: 0.8106786	best: 0.8101830 (162)	total: 19.3s	remaining: 1m 35s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.810183002
bestIteration = 162

Shrink model to first 163 iterations.


CatBoostClassifier(depth=6, iterations=1200, learning_rate=0.03, loss_function='MultiClass', random_seed=42, verbose=100)

In [6]:
X_train.columns

Index(['breed', 'color', 'intake_condition', 'intake_type', 'name',
       'previous_intakes', 'is_repeat', 'intake_hour', 'month_sin',
       'month_cos', 'sex', 'intact', 'is_mixed_breed', 'primary_breed',
       'is_mixed_color', 'primary_color', 'has_name', 'name_length',
       'age_days', 'day_of_week', 'month', 'year', 'is_weekend', 'is_puppy',
       'is_senior', 'age_log', 'puppy_and_intact', 'stray_and_intact',
       'breed_age'],
      dtype='object')

# Evaluation


In [7]:
probs = model.predict_proba(X_test)
y_pred = model.classes_[np.argmax(probs, axis=1)]

#Log Loss
ll = log_loss(y_test, probs, labels=model.classes_)

#Multiclass Brier Score
y_onehot = (
    np.asarray(y_test)[:, None]
    == np.asarray(model.classes_)[None, :]
).astype(int)

brier = np.mean(np.sum((probs - y_onehot) ** 2, axis=1))

#Macro F1
f1 = f1_score(y_test, y_pred, average="macro")

#Balanced Accuracy
bal_acc = balanced_accuracy_score(y_test, y_pred)

print(f"Log Loss:          {ll:.4f}")
print(f"Brier Score:       {brier:.4f}")
print(f"Macro F1:          {f1:.4f}")
print(f"Balanced Accuracy: {bal_acc:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred))

Log Loss:          0.8952
Brier Score:       0.5387
Macro F1:          0.5117
Balanced Accuracy: 0.5397

Classification report:
              precision    recall  f1-score   support

           0       0.52      0.57      0.55      1330
           1       0.56      0.20      0.30      1373
           2       0.59      0.84      0.69      1696

    accuracy                           0.56      4399
   macro avg       0.56      0.54      0.51      4399
weighted avg       0.56      0.56      0.52      4399



In [8]:
from sklearn.metrics import precision_recall_curve, average_precision_score

classes = list(model.classes_)
i2 = classes.index(2)

p2_val = model.predict_proba(X_val)[:, i2]
p2_test = model.predict_proba(X_test)[:, i2]

# Wybór progu na walidacji: najwyższa precyzja przy recall >= 0.6
prec, rec, thr = precision_recall_curve((y_val == 2).astype(int), p2_val)
ok = rec[:-1] >= 0.8
threshold = thr[ok][np.argmax(prec[:-1][ok])]
print(f"Próg dla klasy >30 dni: {threshold:.3f}")

# Ocena na teście
flag = p2_test >= threshold
y_long = (y_test == 2).astype(int)
tp = np.sum(flag & (y_long == 1))
print(f"Oznaczone: {flag.mean():.1%} zwierząt")
print(f"Precyzja: {tp / max(flag.sum(), 1):.3f}")
print(f"Recall:   {tp / y_long.sum():.3f}")
print(f"PR-AUC (klasa >30): {average_precision_score(y_long, p2_test):.3f}")

Próg dla klasy >30 dni: 0.446
Oznaczone: 47.9% zwierząt
Precyzja: 0.642
Recall:   0.798
PR-AUC (klasa >30): 0.721


In [9]:
def predict_cat(model, cat):
    probs = model.predict_proba(cat)[0]

    return pd.DataFrame({
        "Time in shelter": ["≤4 days", "4–10 days", ">30 days"],
        "Probability (%)": [
            round(probs[list(model.classes_).index(i)] * 100, 2)
            for i in [0, 1, 2]
        ]
    })


In [10]:
cat = X_test.iloc[[5]]
print(df_cats.loc[cat.index[0], "time_in_shelter_days"])
display(predict_cat(model, cat))

9.135416666666666


,Time in shelter,Probability (%)
0,≤4 days,31.64
1,4–10 days,46.25
2,>30 days,22.11


In [11]:
cat = X_test.iloc[[30]]

print(df_cats.loc[cat.index[0], "time_in_shelter_days"])

display(predict_cat(model, cat))

9.901388888888889


,Time in shelter,Probability (%)
0,≤4 days,34.49
1,4–10 days,25.67
2,>30 days,39.83


In [12]:
long_stay = X_test[df_cats.loc[X_test.index, "time_in_shelter_days"] > 60]

cat = long_stay.iloc[[0]]

print(df_cats.loc[cat.index[0], "time_in_shelter_days"])
display(predict_cat(model, cat))


62.07986111111112


,Time in shelter,Probability (%)
0,≤4 days,4.85
1,4–10 days,11.47
2,>30 days,83.68
